# FVLC replication: full run from scratch (all 3 protocols)
Everything lives in its own Drive folder (`MyDrive/fvlc_full`). This run is independent of any earlier run.

**Before you start:**
1. In Google Drive, create the folder `fvlc_full` in My Drive.
2. Upload `fvlc-replication.zip` into that folder.
3. In Colab: Runtime → Change runtime type → **T4 GPU**.

The whole run takes about 4–6 h of GPU time. It is resumable: after a disconnect, re-run Step 1 and then Step 3,
and finished parts are skipped. It is fine to split it over 2–3 sessions or days.

## Step 1: Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/fvlc_full'      # the separate folder for this run
ZIP_PATH  = f'{DRIVE_DIR}/fvlc-replication.zip'
assert os.path.exists(ZIP_PATH), f'Upload fvlc-replication.zip to {DRIVE_DIR} first'

if not os.path.exists('/content/fvlc-replication'):
    !unzip -q "$ZIP_PATH" -d /content/
%cd /content/fvlc-replication
!pip install -q -r requirements.txt

os.environ['FVLC_ACT_DIR'] = f'{DRIVE_DIR}/activations'   # cached features for THIS run only
!nvidia-smi -L

## Step 2: Sanity check (~1 min)

In [ ]:
!python tests/test_core.py

## Step 3: Start the full run (in the background)
The run goes CIFAR-10 → CIFAR-100 → CUB. For each dataset it runs:
- protocol A (anchor): Table 1 + Table 2
- protocol B (seat): SEAT-style test-time attack
- protocol C (paired): each method vs itself

The Table 3 ablation runs on CIFAR-10 for all three protocols. The log is saved to Drive.
Run this cell **once**; to resume after a disconnect, run Step 1 and this cell again.

In [ ]:
!nohup bash scripts/run_all.sh $DRIVE_DIR cifar10 cifar100 cub > $DRIVE_DIR/run_all.log 2>&1 &

## Step 4: Check progress (re-run whenever you like)

In [ ]:
!grep -E "^=|^---|^\\[|Traceback|Error|ALL DONE" $DRIVE_DIR/run_all.log | tail -n 25

## Step 5: Results (after the log says ALL DONE)

In [ ]:
from IPython.display import Markdown, Image, display
for P in ['anchor', 'seat', 'paired']:
    t = f'{DRIVE_DIR}/results_{P}/tables.md'
    if os.path.exists(t):
        display(Markdown(f'# Protocol: {P}\n' + open(t).read()))
        display(Image(f'{DRIVE_DIR}/results_{P}/fig_stability.png'))

In [ ]:
# Figure-3-style concept plots (CUB only if its images are present this session)
for ds in ['cifar10', 'cifar100', 'cub']:
    if os.path.exists(f'{DRIVE_DIR}/results_anchor/{ds}/layers/WP1_10+WP2+IP_fvlc.pt') and (ds != 'cub' or os.path.exists('data/CUB')):
        !python visualize.py --dataset $ds --perturbation WP1_10+WP2+IP --index 0 --out $DRIVE_DIR/results_anchor
        display(Image(f'{DRIVE_DIR}/results_anchor/{ds}/fig3_WP1_10+WP2+IP_0.png'))

## Step 6: Download all results (upload this zip to Claude)

In [ ]:
!cd $DRIVE_DIR && zip -qr full_results.zip results_anchor results_seat results_paired run_all.log -x "*/layers/*" "*/vanilla.pt"
from google.colab import files
files.download(f'{DRIVE_DIR}/full_results.zip')